In [ ]:
# Cell 1: Setup and Imports
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import time
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Cell 2: Data Preparation
# For demonstration, we'll use CIFAR-10 (you can replace with your custom dataset)
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

# Load CIFAR-10 dataset
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=100, shuffle=False, num_workers=4)

classes = train_dataset.classes
print(f"Classes: {classes}")

# Cell 3: Model Architecture
class EfficientCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(EfficientCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),
        )
        
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Initialize model
model = EfficientCNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

# Cell 4: Training Function
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    return running_loss / len(loader), 100. * correct / total

def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    return running_loss / len(loader), 100. * correct / total, all_preds, all_labels

# Cell 5: Training Loop
num_epochs = 30
train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc, _, _ = evaluate(model, test_loader, criterion)
    
    scheduler.step(val_loss)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    
    print(f'Epoch {epoch+1}/{num_epochs}:')
    print(f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%')
    print(f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%')
    print('-' * 50)

# Cell 6: Model Evaluation
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
ax1.plot(train_losses, label='Train Loss')
ax1.plot(val_losses, label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.set_title('Training and Validation Loss')

ax2.plot(train_accs, label='Train Acc')
ax2.plot(val_accs, label='Val Acc')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.legend()
ax2.set_title('Training and Validation Accuracy')
plt.tight_layout()
plt.savefig('../training_curves.png')
plt.show()

# Cell 7: Final Evaluation
_, final_acc, all_preds, all_labels = evaluate(model, test_loader, criterion)
print(f'Final Test Accuracy: {final_acc:.2f}%')

# Classification Report
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=classes))

# Confusion Matrix
plt.figure(figsize=(12, 10))
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', xticklabels=classes, yticklabels=classes)
plt.title('Confusion Matrix')
plt.tight_layout()
plt.savefig('../confusion_matrix.png')
plt.show()

# Cell 8: Save Original Model
model_path = Path('../backend/api/models/original')
model_path.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), model_path / 'model_original.pth')
print(f"Original model saved to {model_path}")

# Cell 9: Model Quantization
def quantize_model(model, calibration_loader):
    model.eval()
    
    # Set model to CPU for quantization
    model.cpu()
    
    # Fuse modules for better quantization
    model.eval()
    
    # Prepare for quantization
    model.qconfig = torch.ao.quantization.get_default_qconfig('fbgemm')
    torch.ao.quantization.prepare(model, inplace=True)
    
    # Calibrate with representative dataset
    with torch.no_grad():
        for i, (inputs, _) in enumerate(calibration_loader):
            if i > 100:  # Use subset for calibration
                break
            model(inputs)
    
    # Convert to quantized model
    torch.ao.quantization.convert(model, inplace=True)
    return model

# Create calibration loader
calibration_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)

# Create a copy of the model for quantization
quantized_model = EfficientCNN(num_classes=10)
quantized_model.load_state_dict(torch.load(model_path / 'model_original.pth'))

# Quantize
quantized_model = quantize_model(quantized_model, calibration_loader)

# Cell 10: Save Quantized Model
quantized_path = Path('../backend/api/models/quantized')
quantized_path.mkdir(parents=True, exist_ok=True)
torch.save(quantized_model.state_dict(), quantized_path / 'model_quantized.pth')
print(f"Quantized model saved to {quantized_path}")

# Cell 11: Performance Comparison
def benchmark_inference(model, loader, num_batches=50):
    model.eval()
    if torch.cuda.is_available():
        model = model.cuda()
    
    latencies = []
    with torch.no_grad():
        for i, (inputs, _) in enumerate(loader):
            if i >= num_batches:
                break
            if torch.cuda.is_available():
                inputs = inputs.cuda()
            
            start_time = time.time()
            _ = model(inputs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            latencies.append(time.time() - start_time)
    
    return np.mean(latencies) * 1000  # Return in milliseconds

# Load original model
original_model = EfficientCNN(num_classes=10)
original_model.load_state_dict(torch.load(model_path / 'model_original.pth'))

# Benchmark
print("\nPerformance Comparison:")
print("-" * 50)

# Original model
original_latency = benchmark_inference(original_model, test_loader)
print(f"Original Model - Avg Latency: {original_latency:.2f}ms per batch")

# Quantized model
quantized_latency = benchmark_inference(quantized_model, test_loader)
print(f"Quantized Model - Avg Latency: {quantized_latency:.2f}ms per batch")

latency_reduction = ((original_latency - quantized_latency) / original_latency) * 100
print(f"Latency Reduction: {latency_reduction:.1f}%")

# Save metrics
with open('../performance_metrics.txt', 'w') as f:
    f.write(f"Original Model Latency: {original_latency:.2f}ms\n")
    f.write(f"Quantized Model Latency: {quantized_latency:.2f}ms\n")
    f.write(f"Latency Reduction: {latency_reduction:.1f}%\n")
    f.write(f"Final Test Accuracy: {final_acc:.2f}%\n")

Using device: cpu


100%|██████████████████████████████████████████████████| 170M/170M [00:39<00:00, 4.28MB/s]
C:\Users\saiko\AppData\Local\Programs\Python\Python314\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
Epoch 1/30:
Train Loss: 1.8168, Train Acc: 32.43%
Val Loss: 1.3674, Val Acc: 48.39%
--------------------------------------------------
Epoch 2/30:
Train Loss: 1.5035, Train Acc: 44.90%
Val Loss: 1.1496, Val Acc: 57.25%
--------------------------------------------------
Epoch 3/30:
Train Loss: 1.3421, Train Acc: 51.49%
Val Loss: 1.0296, Val Acc: 62.41%
--------------------------------------------------
Epoch 4/30:
Train Loss: 1.2276, Train Acc: 56.35%
Val Loss: 0.9560, Val Acc: 65.30%
--------------------------------------------------
Epoch 5/30:
Train Loss: 1.1612, Train Acc: 59.09%
Val Loss: 0.8733, Val Acc: 68.87%
--------------------------------------------------
Epoch 6/30:
Train Loss: 1.0968, Train Acc: 61.33%
Val Loss: 0.8552, Val Acc: 69.95%
--------------------------------------------------
Epoch 7/30:
Train Loss: 1.0492, Train Acc: 63.39%
Val Loss: 0.7830, Val Acc: 71.91%
------